# 02 · SFT warm-up (optional) and pass-rate filter

Skip the SFT cell if the spike baseline already had enough mixed groups.

In [ ]:
# --- setup: run once per session -------------------------------------------
import os, subprocess, sys
SPIDER_DIR = "/kaggle/input/spider/spider"   # <- EDIT: the folder with train_spider.json and database/
RUNS = "/kaggle/working/runs"
os.environ["SPIDER_DIR"], os.environ["MIRAGE_RUNS"] = SPIDER_DIR, RUNS
os.chdir("/kaggle/working")
if not os.path.exists("mirage"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SiddharthaRaja/mirage"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "mirage[train]"], check=True)
os.chdir("/kaggle/working/mirage")
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# If you are continuing from an earlier notebook, copy its runs folder in:
# !cp -r /kaggle/input/<your-previous-notebook-output>/runs/* /kaggle/working/runs/ 2>/dev/null || true


In [ ]:
!mirage sft -c configs/sft.yaml

Check the warm-up on **val** (never dev):

In [ ]:
!mirage generate -c configs/grpo_base.yaml --split val --label start
!mirage score $MIRAGE_RUNS/base/eval/start-val/predictions.jsonl --spider $SPIDER_DIR --suites $MIRAGE_RUNS/suites

## Pass-rate filter
Keep training questions where the starting model's attempts disagree under either reward. Both arms use this same file.

In [ ]:
!mirage filter -c configs/grpo_base.yaml --k 8